# S3Eval: Commercial LLM Test (GPT-4o-mini)

Quick test with 22 selected samples (2 per mutation type) to check if stronger models can detect hard mutations.


In [18]:
import sys
import os
import re
import json
import time
import random as _rng
try:
    import resource  # Unix-only
except ImportError:
    resource = None
import platform
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from openai import OpenAI
from jinja2 import Environment, FileSystemLoader
from ipywidgets import interact, IntSlider

from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.model_selection import GroupKFold
from statsmodels.stats.inter_rater import fleiss_kappa as _fleiss_kappa, aggregate_raters

from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

from cve2pddlap.core.data_loader import load_few_shot_pool
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem
from cve2pddlap.llm_providers.remote.openai_compat import QwenProvider

# Project root (relative — works from notebooks/attack_paths/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# Data paths and file names
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
BAD_PDDL_DIR = os.path.join(PROJECT_ROOT, 'resources', 'data', 'mutated_bad_PDDL_AP')
TARGET_POOL_FILE = os.path.join(PROJECT_ROOT, 'resources', 'data', 'target_pool.json')
GENERATED_DOMAIN_DIR = os.path.join(PROJECT_ROOT, 'generated_domain')
FIG_DIR = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_intrinsic')
os.makedirs(FIG_DIR, exist_ok=True)
FIG_DIR_EXT = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_extrinsic')
os.makedirs(FIG_DIR_EXT, exist_ok=True)
EVAL_SET_DIR = os.path.join(GENERATED_DOMAIN_DIR, 'eval_set')
PROMPTS_PATH = os.path.join(PROJECT_ROOT, 'resources', 'prompt', 'evaluation')
AP_PATTERN = re.compile(r'^AP\d+$')
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

# Models (small defaults — replace with preferred models)
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"  # too large for CPU
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"  # Qwen3 embedding model 4B
EMBEDDING_MODEL_NAME_2 = "BAAI/bge-base-en-v1.5"  # fallback: smaller model


LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Generation parameters
SEED = 42
TEMPERATURE = 0.0
TOP_K = 1


@dataclass(frozen=True)
class EvaluationFlags:
    syntax_check: bool = True
    embedding_intrinsic: bool = True
    embedding_extrinsic: bool = True
    llm_intrinsic: bool = True
    llm_extrinsic: bool = True


eval_flags = EvaluationFlags()

os.environ['MallocStackLogging'] = '0'

# Results output directory
RESULTS_BASE = os.path.join(PROJECT_ROOT, "results", "tests", "reference_set")

def get_device_info():
    """Return device info dict."""
    return {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python": platform.python_version(),
        "torch_device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    }


In [19]:
def model_short_name(model_name):
    """Generate a short readable name from a full model identifier.
    e.g. 'meta/llama-3.3-70b-instruct' -> 'llama-3.3-70b'
         'gpt-4.1-mini' -> 'gpt-4.1-mini'
         'all-MiniLM-L6-v2' -> 'all-MiniLM-L6-v2'
         'BAAI/bge-base-en-v1.5' -> 'bge-base-en-v1.5'
         'Qwen/Qwen3-Embedding-4B' -> 'Qwen3-Embedding-4B'
    """
    name = model_name.split("/")[-1]  # strip org prefix
    for suffix in ["-instruct", "-Instruct", "-chat", "-Chat"]:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return name


def load_target_pool(target_pool_file):
    """Load CVE descriptions from target_pool.json. Returns {cve_id: description}."""
    with open(target_pool_file, encoding='utf-8') as f:
        pool = json.load(f)
    return {entry['cve_id']: entry['description'] for entry in pool}


def load_dataset(data_path, cve_descriptions):
    """Load all CVEs with their descriptions (from target_pool) and attack path PDDL files."""
    dataset = []
    for cve_dir in sorted(Path(data_path).iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id)
        if description is None:
            continue
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir() or not AP_PATTERN.match(ap_dir.name):
                continue
            domain_file = ap_dir / DOMAIN_FILE
            problem_file = ap_dir / PROBLEM_FILE
            if domain_file.exists() and problem_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                    'problem': problem_file.read_text(encoding='utf-8').strip(),
                })
        dataset.append({
            'cve_id': cve_id,
            'description': description,
            'attack_paths': attack_paths,
        })
    return dataset


def load_bad_dataset(bad_pddl_dir, cve_descriptions):
    """Load mutated bad PDDL domains from mutated_bad_PDDL_AP/.
    Returns list of {cve_id, description, attack_paths: [{ap_id, domain}]}."""
    bad_dataset = []
    bad_dir = Path(bad_pddl_dir)
    if not bad_dir.exists():
        print(f"WARNING: {bad_pddl_dir} not found")
        return bad_dataset
    for cve_dir in sorted(bad_dir.iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id, "")
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir():
                continue
            domain_file = ap_dir / DOMAIN_FILE
            if domain_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                })
        if attack_paths:
            bad_dataset.append({
                'cve_id': cve_id,
                'description': description,
                'attack_paths': attack_paths,
            })
    return bad_dataset

def load_prompts(prompts_path):
    """Load Jinja2 evaluation prompt templates."""
    return Environment(loader=FileSystemLoader(prompts_path))


def load_embedding_model(model_name):
    """Load a SentenceTransformer bi-encoder model.
    GPU for small models (<2B params), CPU for large models.
    """
    TRUST_REMOTE = ["Qwen3-Embedding", "nomic-ai/", "jinaai/jina-embeddings-v3"]
    # Models known to be too large for GPU encode (>2GB weights)
    FORCE_CPU = ["Qwen3-Embedding-4B", "Qwen3-Embedding-8B", "jina-embeddings-v3", "stella_en_1.5B"]
    kwargs = {"trust_remote_code": True} if any(t in model_name for t in TRUST_REMOTE) else {}
    if any(t in model_name for t in FORCE_CPU):
        model = SentenceTransformer(model_name, device="cpu", **kwargs)
        print(f"  Loaded {model_name} on CPU (large model)")
        return model
    try:
        model = SentenceTransformer(model_name, **kwargs)
        print(f"  Loaded {model_name} on {model.device}")
        return model
    except (RuntimeError, torch.cuda.OutOfMemoryError):
        import gc; gc.collect(); torch.cuda.empty_cache()
        model = SentenceTransformer(model_name, device="cpu", **kwargs)
        print(f"  GPU OOM, loaded {model_name} on CPU")
        return model

def load_llm(model_name):
    """Load a HuggingFace causal LLM with its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    gen = hf_pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=False,
    )
    return gen, tokenizer



def strip_base_score(cvss_nl):
    """Remove Base Score from CVSS vector NL string."""
    if not cvss_nl:
        return ""
    return _re.sub(r"\s*Base Score:\s*[\d.]+\.?\s*", "", cvss_nl).strip()

def build_nl_ti_selected(entry_ti):
    """Build TI-selected NL: desc + capec + cvss_vector_nl (no base score)."""
    parts = [entry_ti.get("description", "")]
    capec = entry_ti.get("capec", "")
    if capec:
        parts.append(capec)
    cvss = strip_base_score(entry_ti.get("cvss_vector_nl", ""))
    if cvss:
        parts.append(cvss)
    return " ".join(parts)

cve_descriptions = load_target_pool(TARGET_POOL_FILE)
dataset = load_dataset(DATASET_PATH, cve_descriptions)
bad_dataset = load_bad_dataset(BAD_PDDL_DIR, cve_descriptions)
print(f"Bad (mutated) examples: {len(bad_dataset)} CVEs, {sum(len(e['attack_paths']) for e in bad_dataset)} domains")
prompt_env = load_prompts(PROMPTS_PATH)


embedding_model = None
if eval_flags.embedding_intrinsic or eval_flags.embedding_extrinsic:
    embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

llm, tokenizer = None, None
if eval_flags.llm_intrinsic or eval_flags.llm_extrinsic:
    llm, tokenizer = load_llm(LLM_MODEL_NAME)

# --- Calibration data for LLM-as-expert evaluation ---
CALIBRATION_DATA_PATH = os.path.join(PROMPTS_PATH, "data.jsonl")

def load_calibration_data(data_jsonl_path, eval_type="intrinsic", exclude_cve=None, n=4, seed=42):
    """Sample n calibration examples from data.jsonl.
    
    Constraints:
        - exclude_cve: the CVE being evaluated is excluded (prevent data leakage)
        - n >= 2: at least 1 good + 1 bad example guaranteed
        - balanced: samples from both calibration_good and calibration_bad pools
    
    Args:
        data_jsonl_path: path to data.jsonl
        eval_type: 'intrinsic' or 'extrinsic'
        exclude_cve: CVE ID to exclude
        n: total number of calibration examples (>= 2)
        seed: random seed for reproducibility
    """
    rng = _rng.Random(seed)
    
    with open(data_jsonl_path) as f:
        all_data = [json.loads(line) for line in f]
    
    pool = [d for d in all_data
            if d.get("eval_type") == eval_type
            and d.get("role", "").startswith("calibration")
            and d.get("cve_id") != exclude_cve]
    
    good = [d for d in pool if d.get("role") == "calibration_good"]
    bad = [d for d in pool if d.get("role") == "calibration_bad"]
    
    n = max(n, 2)  # enforce minimum 2
    n_good = max(1, n // 2)       # at least 1 good
    n_bad = max(1, n - n_good)    # at least 1 bad
    # Adjust if one pool is too small
    n_good = min(n_good, len(good))
    n_bad = min(n_bad, len(bad))
    
    selected_good = rng.sample(good, n_good) if good else []
    selected_bad = rng.sample(bad, n_bad) if bad else []
    
    return selected_good + selected_bad


# --- Rate limit retry wrapper ---
def api_call_with_retry(func, *args, max_retries=5, base_delay=5, **kwargs):
    """Call func with exponential backoff on rate limit errors."""
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [RATE LIMIT] retry {attempt+1}/{max_retries} in {delay}s...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")


# --- Classification report helpers (following Marco's pattern) ---

def report_row(y_true, y_pred, **meta):
    """Flatten classification_report into a single dict row, with TPR/FPR."""
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    row = dict(meta)
    for key, val in rpt.items():
        if isinstance(val, dict):
            for metric, v in val.items():
                row[f'{key}__{metric}'] = v
        else:
            row[key] = val
    row['tpr'] = rpt.get('1', {}).get('recall', float('nan'))
    row['fpr'] = 1.0 - rpt.get('0', {}).get('recall', float('nan'))
    return row

def print_report(y_true, y_pred, title=''):
    if title:
        print(f'\n{title}')
        print('─' * len(title))
    print(classification_report(y_true, y_pred, zero_division=0))


def save_cv_record(cv_path, model_name, threshold, fold_thresholds, row, labels, build_seconds, cv_seconds, cv_method="reference_full_matrix"):
    """Build CV record, dedup by model, and save to JSONL."""
    record = {
        "timestamp": datetime.now().isoformat(),
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "fold_thresholds": fold_thresholds,
        "accuracy": row.get("accuracy", float("nan")),
        "precision": row.get("1__precision", float("nan")),
        "recall": row.get("1__recall", float("nan")),
        "f1": row.get("1__f1-score", float("nan")),
        "tpr": row["tpr"],
        "fpr": row["fpr"],
        "n_positive_pairs": int(labels.sum()),
        "n_negative_pairs": int((labels == 0).sum()),
        "build_pairs_seconds": build_seconds,
        "cv_calibration_seconds": cv_seconds,
    }
    existing = []
    if os.path.exists(cv_path):
        with open(cv_path) as f:
            existing = [json.loads(line) for line in f if line.strip()]
        existing = [r for r in existing if not (r.get("model") == model_name and r.get("cv_method", "reference_full_matrix") == cv_method)]
    existing.append(record)
    with open(cv_path, "w") as f:
        for r in existing:
            f.write(json.dumps(r) + "\n")
    return record


def save_intrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, ap_id, similarity, prediction, elapsed, cv_method="reference_full_matrix"):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, test_ap, best_ref_ap, similarity, prediction, n_references, elapsed):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "cve_id": cve_id,
        "test_ap": test_ap,
        "best_ref_ap": best_ref_ap,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "n_references": n_references,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "label": label, "response_length": len(response),
        "parse_success": label is not None,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, scores, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "verdict": verdict, "final_score": final_score, "scores": scores,
        "response_length": len(response),
        "parse_success": "parse_error" not in scores,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "label": label, "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "verdict": verdict, "final_score": final_score,
        "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def prepare_intrinsic_texts(dataset):
    """Extract texts and build 21x55 pair structure (matches build_intrinsic_pairs).
    Returns:
        descs: 21 unique CVE descriptions (one per CVE)
        domains: 55 reference domains (one per attack path)
        domain_cve_ids: 55 CVE IDs (domain-side, kept for backward compatibility)
        labels: 1155 pair labels (1 if same CVE, else 0), iterated descs-major
        groups: 1155 description-side CVE IDs (for GroupKFold)
    """
    descs = [entry["description"] for entry in dataset]
    desc_cve_ids = [entry["cve_id"] for entry in dataset]
    domains, domain_cve_ids = [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            domains.append(ap["domain"])
            domain_cve_ids.append(entry["cve_id"])
    n_d, n_p = len(descs), len(domains)
    labels = np.array([1 if desc_cve_ids[i] == domain_cve_ids[j] else 0
                       for i in range(n_d) for j in range(n_p)])
    groups = np.array([desc_cve_ids[i] for i in range(n_d) for j in range(n_p)])
    return descs, domains, domain_cve_ids, labels, groups


def compute_intrinsic_scores(descs, domains, model):
    """Compute 21x55 similarity scores (flattened, descs-major)."""
    sim_matrix = embedding_similarity_intrinsic(descs, domains, model)
    n_d, n_p = len(descs), len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n_d) for j in range(n_p)])
    return scores


def prepare_extrinsic_texts(dataset):
    """Extract texts and build pair structure for extrinsic (model-independent)."""
    all_entries = []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            all_entries.append((ap["domain"], entry["cve_id"], ap["ap_id"]))
    domains = [p[0] for p in all_entries]
    cve_ids = [p[1] for p in all_entries]
    n = len(all_entries)
    labels = np.array([1 if cve_ids[i] == cve_ids[j] else 0 for i in range(n) for j in range(i+1, n)])
    groups = np.array([cve_ids[i] for i in range(n) for j in range(i+1, n)])
    return domains, cve_ids, labels, groups


def compute_extrinsic_scores(domains, model):
    """Compute similarity scores for pre-prepared extrinsic texts."""
    E = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E @ E.T).float().cpu().numpy()
    n = len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n) for j in range(i+1, n)])
    return scores


Bad (mutated) examples: 18 CVEs, 55 domains
  Loaded all-MiniLM-L6-v2 on cpu


Device set to use cpu


In [28]:
# ── Client Setup ──
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    timeout=180.0,
)
MODELS = ['gpt-5.5']
CLIENT = client

# Test
try:
    test = CLIENT.chat.completions.create(model=MODELS[0], messages=[{'role': 'user', 'content': 'Say OK'}], max_completion_tokens=5)
    print(f'Connection OK: {test.choices[0].message.content}')
except Exception as e:
    print(f'FAILED: {e}')


Connection OK: OK


In [21]:
# ── Response parsing helpers ──
SCORED_CRITERIA = ['F1','F2','A1','A2','C1','C2','C3','V1','V2','V3','N1','N2']
SCORED_CRITERIA_EXT = SCORED_CRITERIA + ['R1','R2','R3']

def parse_scored_response(response_text):
    """Parse LLM scored response JSON, extract scores."""
    try:
        text = response_text.strip()
        if text.startswith('```'):
            text = text.split('\n', 1)[1]
            text = text.rsplit('```', 1)[0]
        data = json.loads(text)
        return data
    except Exception:
        return None

def domain_min_score(scores, criteria=SCORED_CRITERIA):
    """Return min of valid criteria scores (0-5). None if no valid scores."""
    valid = [scores[k] for k in criteria if isinstance(scores.get(k), (int, float))]
    return min(valid) if valid else None

def parse_binary_response(response_text):
    """Parse binary True/False from LLM response."""
    try:
        text = response_text.strip()
        if text.startswith('```'): text = text.split('\n', 1)[1].rsplit('```', 1)[0]
        raw_label = json.loads(text).get('label', '?')
        return True if str(raw_label).lower() in ('true', '1', 'yes') else False
    except Exception:
        # Fallback: check raw text
        t = response_text.strip().lower()
        if 'true' in t: return True
        if 'false' in t: return False
        return None

print('Parse functions loaded')


Parse functions loaded


In [22]:
# Unified evaluation template (binary/scored × intrinsic/extrinsic via parameters)
eval_template = prompt_env.get_template("completion.md.jinja")

def llm_eval_intrinsic_binary(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """Binary True/False classification: does the PDDL match the CVE?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=4096,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [23]:
def llm_eval_intrinsic_scored(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """12-criteria scored evaluation (integer 0-5, violation/non-violation scale)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=4096,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [24]:
def llm_eval_extrinsic_binary(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """Binary True/False: does the candidate match the CVE and reference?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=4096,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [25]:
def llm_eval_extrinsic_scored(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """16-criteria scored evaluation (12 quality + 4 reference-comparison, integer 0-5)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=4096,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [26]:
# ── Selected 22 bad samples (2 per mutation type) ──
# ── 7 samples: 5 hardest (GPT-4.1 extrinsic failed) + 2 controls ──
SELECTED_BAD = [
    # Hardest (GPT-4.1 extrinsic: target criterion = 5, undetected)
    ('CVE-2024-38816', 'AP1_delete_random_precondition_rep1', 'AP1', 'delete_random_precondition'),
    ('CVE-2024-22262', 'AP1_swap_action_effects_rep3', 'AP1', 'swap_action_effects'),
    ('CVE-2022-40150', 'AP1_replace_stride_goal_rep2', 'AP1', 'replace_stride_goal'),
    ('CVE-2025-24813', 'AP1_scramble_action_names_rep0', 'AP1', 'scramble_action_names'),
    ('CVE-2022-40149', 'AP2_replace_with_abstract_action_rep4', 'AP2', 'replace_with_abstract_action'),
    # Controls (GPT-4.1 extrinsic detected)
    ('CVE-2023-33202', 'AP3_inject_capability_violation_rep3', 'AP3', 'inject_capability_violation'),
    ('CVE-2022-40149', 'AP1_corrupt_exposure_action_rep1', 'AP1', 'corrupt_exposure_action'),
]
print(f'Selected: {len(SELECTED_BAD)} bad samples covering {len(set(m for _,_,_,m in SELECTED_BAD))} mutation types')

# Build lookup for reference domains
ref_lookup = {}
for entry in dataset:
    for ap in entry['attack_paths']:
        ref_lookup[(entry['cve_id'], ap['ap_id'])] = {
            'domain': ap['domain'],
            'description': entry['description'],
        }

# Build lookup for bad domains
bad_lookup = {}
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        bad_lookup[(entry['cve_id'], ap['ap_id'])] = {
            'domain': ap['domain'],
            'description': entry['description'],
        }

print(f'Reference lookup: {len(ref_lookup)} entries')
print(f'Bad lookup: {len(bad_lookup)} entries')


Selected: 7 bad samples covering 7 mutation types
Reference lookup: 55 entries
Bad lookup: 55 entries


## Intrinsic Scored Evaluation


In [17]:
# ── Run Intrinsic Scored on selected samples ──
save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
os.makedirs(save_dir, exist_ok=True)

SCORED_CRITERIA = ['F1','F2','A1','A2','C1','C2','C3','V1','V2','V3','N1','N2']

mut_to_criterion = {
    'delete_critical_action': 'C2',
    'delete_random_precondition': 'C1',
    'swap_action_effects': 'A2',
    'replace_stride_goal': 'V3',
    'corrupt_exposure_action': 'V1',
    'merge_consecutive_actions': 'A1',
    'inject_capability_violation': 'F1',
    'replace_with_abstract_action': 'F2',
    'replace_exploitation_mechanism': 'V2',
    'scramble_action_names': 'N1',
    'scramble_predicate_names': 'N2',
}

for model_name in MODELS:
    print(f'\n{"="*60}')
    print(f'Intrinsic Scored: {model_name}')
    print(f'{"="*60}')
    
    results_path = os.path.join(save_dir, f'results_intrinsic_scored_test_{model_short_name(model_name)}.jsonl')
    t_start = time.time()
    
    # # 1. Selected reference samples (expected: True)
    # print(f'\n--- Reference (expected: True) ---')
    # ref_cves_done = set()
    # for cve_id, bad_ap_id, source_ap, mut_type in SELECTED_BAD:
    #     key = (cve_id, source_ap)
    #     if key in ref_cves_done: continue
    #     ref_cves_done.add(key)
    #     ref = ref_lookup.get(key)
    #     if ref is None: continue
    #     try:
    #         response, usage = llm_eval_intrinsic_scored(
    #             cve_id, ref['description'], ref['domain'],
    #             CLIENT, model_name, seed=42, n_calibration=2)
    #         scores = parse_scored_response(response) or {'parse_error': True}
    #         final_score = domain_min_score(scores) if 'parse_error' not in scores else None
    #         verdict = True if final_score is not None and final_score >= 3 else False
    #         scores_str = ' '.join(f'{c}={scores.get(c,"?")}' for c in SCORED_CRITERIA)
    #         print(f'  [ref] {cve_id}/{source_ap}  verdict={verdict}  min={final_score}  {scores_str}')
    #         save_intrinsic_scored_result(results_path, model_name, 42, 0.0, 'reference', cve_id, source_ap, verdict, final_score, scores, response, usage, 0, 0)
    #     except Exception as e:
    #         print(f'  [ref] {cve_id}/{source_ap}  ERROR: {e}')
    
    # 2. Selected bad samples (expected: False)
    print(f'\n--- Bad (expected: False) ---')
    for cve_id, bad_ap_id, source_ap, mut_type in SELECTED_BAD:
        bad = bad_lookup.get((cve_id, bad_ap_id))
        if bad is None:
            print(f'  [bad] {cve_id}/{bad_ap_id}  SKIP: not found')
            continue
        target_crit = mut_to_criterion.get(mut_type, '?')
        try:
            response, usage = llm_eval_intrinsic_scored(
                cve_id, bad['description'], bad['domain'],
                CLIENT, model_name, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            target_score = scores.get(target_crit, '?')
            scores_str = ' '.join(f'{c}={scores.get(c,"?")}' for c in SCORED_CRITERIA)
            detected = '✓' if (isinstance(target_score, (int,float)) and target_score <= 2) else '✗'
            print(f'  [bad] {cve_id}/{bad_ap_id}  [{mut_type}] target={target_crit}={target_score} {detected}  verdict={verdict}  min={final_score}  {scores_str}')
            save_intrinsic_scored_result(results_path, model_name, 42, 0.0, 'bad', cve_id, bad_ap_id, verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{bad_ap_id}  ERROR: {e}')
    
    elapsed = time.time() - t_start
    print(f'\nDone: {model_name} | {elapsed:.1f}s')



Intrinsic Scored: deepseek-v4-pro

--- Bad (expected: False) ---
  [bad] CVE-2024-38816/AP1_delete_random_precondition_rep1  [delete_random_precondition] target=C1=5 ✗  verdict=False  min=1  F1=5 F2=5 A1=5 A2=1 C1=5 C2=5 C3=5 V1=5 V2=5 V3=5 N1=5 N2=5
  [bad] CVE-2024-22262/AP1_swap_action_effects_rep3  [swap_action_effects] target=A2=1 ✓  verdict=False  min=1  F1=5 F2=4 A1=5 A2=1 C1=1 C2=5 C3=5 V1=5 V2=5 V3=5 N1=5 N2=5
  [bad] CVE-2022-40150/AP1_replace_stride_goal_rep2  [replace_stride_goal] target=V3=4 ✗  verdict=False  min=1  F1=5 F2=5 A1=5 A2=5 C1=1 C2=4 C3=5 V1=5 V2=5 V3=4 N1=5 N2=5
  [bad] CVE-2025-24813/AP1_scramble_action_names_rep0  [scramble_action_names] target=N1=1 ✓  verdict=False  min=1  F1=5 F2=5 A1=5 A2=5 C1=5 C2=5 C3=5 V1=5 V2=5 V3=5 N1=1 N2=5
  [bad] CVE-2022-40149/AP2_replace_with_abstract_action_rep4  [replace_with_abstract_action] target=F2=5 ✗  verdict=False  min=2  F1=5 F2=5 A1=5 A2=5 C1=5 C2=5 C3=5 V1=5 V2=5 V3=5 N1=2 N2=2
  [bad] CVE-2023-33202/AP3_inject_capa

## Analysis


In [ ]:
# ── Analyze results: per-mutation detection rate ──
for model_name in MODELS:
    results_path = os.path.join(save_dir, f'results_intrinsic_scored_test_{model_short_name(model_name)}.jsonl')
    if not os.path.exists(results_path): continue
    
    print(f'\n=== {model_name} ===')
    print(f'{"Mutation":<35} {"Target":<6} {"Score":<6} {"Detected":<8} {"Verdict"}')
    print('-' * 70)
    
    detected_count = 0
    total_bad = 0
    
    with open(results_path) as f:
        for line in f:
            if not line.strip(): continue
            r = json.loads(line)
            if r.get('source') != 'bad': continue
            total_bad += 1
            ap_id = r.get('ap_id', '')
            scores = r.get('scores', {})
            verdict = r.get('verdict')
            
            for mt, crit in mut_to_criterion.items():
                if mt in ap_id:
                    target_score = scores.get(crit, '?')
                    detected = isinstance(target_score, (int,float)) and target_score <= 2
                    if detected: detected_count += 1
                    print(f'{mt:<35} {crit:<6} {str(target_score):<6} {"✓" if detected else "✗":<8} {verdict}')
                    break
    
    print(f'\nDetection rate: {detected_count}/{total_bad} = {detected_count/total_bad:.0%}' if total_bad > 0 else '')


## Extrinsic Scored Evaluation


In [ ]:
# # ── Run Extrinsic Scored on selected bad samples ──
# save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
# os.makedirs(save_dir_ext, exist_ok=True)

# SCORED_CRITERIA_EXT = ['F1','F2','A1','A2','C1','C2','C3','V1','V2','V3','N1','N2','R1','R2','R3']

# for model_name in MODELS:
#     print(f'\n{"="*60}')
#     print(f'Extrinsic Scored: {model_name}')
#     print(f'{"="*60}')
    
#     results_path = os.path.join(save_dir_ext, f'results_extrinsic_scored_test_{model_short_name(model_name)}.jsonl')
    
#     print(f'\n--- Bad vs Source AP (expected: False) ---')
#     detected_count = 0
#     total = 0
#     for cve_id, bad_ap_id, source_ap, mut_type in SELECTED_BAD:
#         bad = bad_lookup.get((cve_id, bad_ap_id))
#         ref = ref_lookup.get((cve_id, source_ap))
#         if bad is None or ref is None:
#             print(f'  SKIP: {cve_id}/{bad_ap_id}')
#             continue
#         target_crit = mut_to_criterion.get(mut_type, '?')
#         try:
#             response, usage = llm_eval_extrinsic_scored(
#                 cve_id, ref['description'], ref['domain'], bad['domain'],
#                 CLIENT, model_name, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             target_score = scores.get(target_crit, '?')
#             detected = isinstance(target_score, (int,float)) and target_score <= 2
#             if detected: detected_count += 1
#             total += 1
#             marker = '✓' if detected else '✗'
#             scores_str = ' '.join(f'{c}={scores.get(c,"?")}' for c in SCORED_CRITERIA_EXT)
#             print(f'  [{mut_type}] target={target_crit}={target_score} {marker}  verdict={verdict}  min={final_score}  {scores_str}')
#             save_extrinsic_scored_result(results_path, model_name, 42, 0.0, 'bad', cve_id, bad_ap_id, source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, 0, 0)
#         except Exception as e:
#             print(f'  [{mut_type}] {cve_id}/{bad_ap_id}  ERROR: {e}')
    
#     print(f'\nDetection rate: {detected_count}/{total} = {detected_count/total:.0%}' if total > 0 else '')


## Extrinsic vs Intrinsic Comparison


In [ ]:
# # ── Compare intrinsic vs extrinsic detection ──
# print(f'{"Mutation":<35} {"Target":<6} {"Intr":<6} {"Extr":<6}')
# print('-' * 60)

# for model_name in MODELS:
#     intr_path = os.path.join(save_dir, f'results_intrinsic_scored_test_{model_short_name(model_name)}.jsonl')
#     extr_path = os.path.join(save_dir_ext, f'results_extrinsic_scored_test_{model_short_name(model_name)}.jsonl')
    
#     # Load intrinsic scores for bad samples
#     intr_scores = {}
#     if os.path.exists(intr_path):
#         with open(intr_path) as f:
#             for line in f:
#                 if not line.strip(): continue
#                 r = json.loads(line)
#                 if r.get('source') == 'bad':
#                     intr_scores[r['ap_id']] = r.get('scores', {})
    
#     # Load extrinsic scores for bad samples
#     extr_scores = {}
#     if os.path.exists(extr_path):
#         with open(extr_path) as f:
#             for line in f:
#                 if not line.strip(): continue
#                 r = json.loads(line)
#                 if r.get('source') == 'bad':
#                     extr_scores[r['ap_id']] = r.get('scores', {})
    
#     print(f'\n=== {model_name} ===')
#     for cve_id, bad_ap_id, source_ap, mut_type in SELECTED_BAD:
#         target_crit = mut_to_criterion.get(mut_type, '?')
#         i_score = intr_scores.get(bad_ap_id, {}).get(target_crit, '?')
#         e_score = extr_scores.get(bad_ap_id, {}).get(target_crit, '?')
#         i_mark = '✓' if isinstance(i_score, (int,float)) and i_score <= 2 else '✗'
#         e_mark = '✓' if isinstance(e_score, (int,float)) and e_score <= 2 else '✗'
#         print(f'{mut_type:<35} {target_crit:<6} {i_score}{i_mark:<5} {e_score}{e_mark:<5}')


## Test: Reference samples (GPT-5.4)


In [29]:
# ── Test reference samples: intrinsic scored + binary ──
# Pick 3 reference samples
test_refs = [
    ('CVE-2022-1471', 'AP1'),
    ('CVE-2022-40149', 'AP1'),
    ('CVE-2023-44487', 'AP1'),
]

print('=== Intrinsic SCORED (reference, expected: True) ===')
for cve_id, ap_id in test_refs:
    ref = ref_lookup.get((cve_id, ap_id))
    if ref is None: continue
    response, usage = llm_eval_intrinsic_scored(
        cve_id, ref['description'], ref['domain'],
        CLIENT, MODELS[0], n_calibration=2)
    scores = parse_scored_response(response) or {'parse_error': True}
    final_score = domain_min_score(scores) if 'parse_error' not in scores else None
    verdict = True if final_score is not None and final_score >= 3 else False
    scores_str = ' '.join(f'{c}={scores.get(c,"?")}' for c in SCORED_CRITERIA)
    print(f'  [scored] {cve_id}/{ap_id}  verdict={verdict}  min={final_score}  {scores_str}')

print('\n=== Intrinsic BINARY (reference, expected: True) ===')
for cve_id, ap_id in test_refs:
    ref = ref_lookup.get((cve_id, ap_id))
    if ref is None: continue
    response, usage = llm_eval_intrinsic_binary(
        cve_id, ref['description'], ref['domain'],
        CLIENT, MODELS[0], n_calibration=2)
    label = parse_binary_response(response)
    print(f'  [binary] {cve_id}/{ap_id}  label={label}  response={repr(response[:100])}')


=== Intrinsic SCORED (reference, expected: True) ===
  [scored] CVE-2022-1471/AP1  verdict=True  min=3  F1=5 F2=5 A1=4 A2=4 C1=4 C2=5 C3=5 V1=3 V2=3 V3=5 N1=4 N2=4
  [scored] CVE-2022-40149/AP1  verdict=True  min=3  F1=5 F2=5 A1=4 A2=3 C1=4 C2=5 C3=5 V1=4 V2=4 V3=5 N1=5 N2=4
  [scored] CVE-2023-44487/AP1  verdict=True  min=4  F1=5 F2=5 A1=4 A2=4 C1=4 C2=4 C3=5 V1=4 V2=5 V3=5 N1=4 N2=4

=== Intrinsic BINARY (reference, expected: True) ===
  [binary] CVE-2022-1471/AP1  label=True  response='{"label": true}'
  [binary] CVE-2022-40149/AP1  label=True  response='{"label": true}'
  [binary] CVE-2023-44487/AP1  label=True  response='{"label": true}'
